In [2]:
!pip3 install boto3


  Obtaining dependency information for boto3 from https://files.pythonhosted.org/packages/d3/87/8189f22ee798177bc7b40afd13f046442c5f91b699e70a950b42ff447e80/boto3-1.37.38-py3-none-any.whl.metadata
  Obtaining dependency information for botocore<1.38.0,>=1.37.38 from https://files.pythonhosted.org/packages/55/1b/93f3504afc7c523dcaa8a8147cfc75421983e30b08d9f93a533929589630/botocore-1.37.38-py3-none-any.whl.metadata
  Obtaining dependency information for jmespath<2.0.0,>=0.7.1 from https://files.pythonhosted.org/packages/31/b4/b9b800c45527aadd64d5b442f9b932b00648617eb5d63d2c7a6587b7cafc/jmespath-1.0.1-py3-none-any.whl.metadata
  Obtaining dependency information for s3transfer<0.12.0,>=0.11.0 from https://files.pythonhosted.org/packages/45/39/13402e323666d17850eca87e4cd6ecfcf9fd7809cac9efdcce10272fc29d/s3transfer-0.11.5-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 60.9 MB/s

SyntaxError: invalid syntax (3367382171.py, line 1)

In [ ]:
import boto3
import time
import os

# ------------------------
# CONFIGURATION
# ------------------------
region = "eu-west-2"  # must be a region that supports Transcribe
bucket_name = "mdt-meeting"  # replace with your S3 bucket
audio_file = "mic_pitch_1_dropout_0.wav"           # your local audio file (wav or mp3)
job_name = "mdttest"            # must be unique per run

# ------------------------
# INIT AWS CLIENTS
# ------------------------
s3 = boto3.client("s3", region_name=region)

# ------------------------
# UPLOAD FILE TO S3
# ------------------------
#print("Uploading {} to S3...".format(audio_file))
#s3.upload_file(audio_file, bucket_name, os.path.basename(audio_file))

media_uri = "s3://mdt-meeting/mic_pitch_1_dropout_0.wav"


transcribe = boto3.client(
    "transcribe",
    region_name=region,
    aws_access_key_id=<AWS_ACCESS_KEY>,
    aws_secret_access_key=<AWS_SECRET_ACCESS_KEY>
)

# ------------------------
# START TRANSCRIBE JOB
# ------------------------
print("Starting Transcribe job...")
transcribe.get_medical_transcription_job(
    TranscriptionJobName=job_name,
    Media={"MediaFileUri": media_uri},
    MediaFormat="wav",          # or "mp3"
    LanguageCode="en-GB",       # change if needed
    OutputBucketName=bucket_name  # saves transcript JSON to same bucket
)

# ------------------------
# WAIT FOR COMPLETION
# ------------------------
while True:
    status = transcribe.get_transcription_job(TranscriptionJobName=job_name)
    state = status["TranscriptionJob"]["TranscriptionJobStatus"]
    if state in ["COMPLETED", "FAILED"]:
        break
    print("Job in progress... waiting 10s")
    time.sleep(10)

# ------------------------
# FETCH RESULT
# ------------------------
if state == "COMPLETED":
    transcript_uri = status["TranscriptionJob"]["Transcript"]["TranscriptFileUri"]
    print("Transcript ready! Download from:")
    print(transcript_uri)
else:
    print("Transcription job failed.")

Starting Transcribe job...


ParamValidationError: Parameter validation failed:
Missing required parameter in input: "MedicalTranscriptionJobName"
Unknown parameter in input: "TranscriptionJobName", must be one of: MedicalTranscriptionJobName
Unknown parameter in input: "Media", must be one of: MedicalTranscriptionJobName
Unknown parameter in input: "MediaFormat", must be one of: MedicalTranscriptionJobName
Unknown parameter in input: "LanguageCode", must be one of: MedicalTranscriptionJobName
Unknown parameter in input: "OutputBucketName", must be one of: MedicalTranscriptionJobName

In [ ]:
import os
import time
import boto3
import pandas as pd
from urllib.parse import urljoin
import requests
import json
from urllib.parse import urlparse
from tqdm import tqdm 


# --- Configuration ---
AWS_REGION = "eu-west-2"  # change if needed
S3_BUCKET = "mdt-meeting"  # must exist already
S3_OUTPUT = 'mdt-output'

#TRANSCRIBE_ROLE_ARN = "arn:aws:iam::<your-account-id>:role/<your-transcribe-role>"  # optional if using default credentials

# Initialize AWS clients
s3 = boto3.client("s3", region_name=AWS_REGION, aws_access_key_id=<access_key>, aws_secret_access_key=<accesss_secret>)
transcribe = boto3.client("transcribe", region_name=AWS_REGION, aws_access_key_id=<AWS_ACCESS_KEY>, aws_secret_access_key=<AWS_SECRET_ACCESS_KEY>)

def upload_to_s3(file_path, bucket):
    """Uploads local audio file to S3 and returns the S3 URI."""
    try:
        file_name = os.path.basename(file_path)
        s3.upload_file(file_path, bucket, file_name)
        s3_uri = f"s3://{bucket}/{file_name}"
        return s3_uri
    except FileNotFoundError:
        print(f"⚠️ File not found: {file_path}")
    except NoCredentialsError:
        print("❌ No AWS credentials found while uploading to S3.")
    except ClientError as e:
        print(f"❌ S3 upload error: {e}")
    return None

def start_transcription_job(job_name, s3_uri, output_bucket):
    """Starts an Amazon Transcribe Medical job."""
    try:
        transcribe.start_medical_transcription_job(
            MedicalTranscriptionJobName=job_name,
            Media={"MediaFileUri": s3_uri},
            OutputBucketName=output_bucket,
            LanguageCode="en-US",  # Amazon Transcribe Medical supports only English
            Specialty="PRIMARYCARE",
            Type="CONVERSATION"  # or "DICTATION"
        )
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConflictException":
            print(f"⚠️ Job name already exists: {job_name}. Skipping...")
        else:
            print(f"❌ Error starting transcription job {job_name}: {e}")
    except Exception as e:
        print(f"❌ Unexpected error starting job {job_name}: {e}")
    return False

def wait_for_completion(job_name, max_wait_minutes=30):
    """Waits for the transcription job to complete and returns the result URI."""
    start_time = time.time()
    while True:
        try:
            status = transcribe.get_medical_transcription_job(
                MedicalTranscriptionJobName=job_name
            )
            job = status["MedicalTranscriptionJob"]
            job_status = job["TranscriptionJobStatus"]

            if job_status in ["COMPLETED", "FAILED"]:
                return job
        except EndpointConnectionError:
            print("⚠️ Network issue while checking job status. Retrying...")
        except ClientError as e:
            print(f"❌ Error getting job status: {e}")
            return None

        if (time.time() - start_time) > max_wait_minutes * 60:
            print(f"⚠️ Timeout waiting for job {job_name}.")
            return None

        time.sleep(10)

def get_transcript_from_uri(output_uri):
    """Fetch transcript text from Amazon's transcript URI."""
    try:
        response = requests.get(output_uri, timeout=30)
        response.raise_for_status()
        data = response.json()
        if "results" in data and "transcripts" in data["results"]:
            return data["results"]["transcripts"][0]["transcript"]
        else:
            return "[ERROR: Transcript JSON missing expected keys]"
    except requests.exceptions.RequestException as e:
        return f"[ERROR: Failed to fetch transcript URI: {e}]"
    except json.JSONDecodeError:
        return "[ERROR: Invalid JSON format from AWS]"
    except Exception as e:
        return f"[ERROR: Unexpected issue reading transcript: {e}]"
    
def get_transcript_from_s3_bucket(transcript_uri, bucket):
    """Fetch transcript JSON directly from your S3 bucket (no public URL)."""
    try:
        # Extract file key from the URI (e.g. s3:// or https:// URL)
        parsed = urlparse(transcript_uri)
        #print (parsed)
        if parsed.scheme.startswith("http"):
            # URL format: https://s3.<region>.amazonaws.com/<bucket>/<key>
            key = "/".join(parsed.path.lstrip("/").split("/")[1:])
        else:
            # S3 URI format: s3://bucket/key
            key = parsed.path.lstrip("/")

        s3_object = s3.get_object(Bucket=bucket, Key=key)
        content = s3_object["Body"].read().decode("utf-8")
        data = json.loads(content)
        return data["results"]["transcripts"][0]["transcript"]
    except Exception as e:
        return f"[ERROR: Unable to read transcript from S3 bucket: {e}]"

    
    
# --- Main ---


for i in list(range(1,11)):
    results = []
    AUDIO_FOLDER = f"augmented_audio/MDT_meeting_full_case{i}"
    OUTPUT_CSV = f"outputs/AWS_Transcript_2/MDT_meeting_full_case{i}.csv"

    for file_name in tqdm(os.listdir(AUDIO_FOLDER)):
        #print (file_name)
        if not file_name.lower().endswith((".mp3", ".wav", ".flac", ".mp4")):
            continue

        local_path = os.path.join(AUDIO_FOLDER, file_name)
        job_name = f"medical-{os.path.splitext(file_name)[0]}-{int(time.time())}"

        #print(f"\n🎧 Processing: {file_name}")

        # Upload file
        s3_uri = upload_to_s3(local_path, S3_BUCKET)
        if not s3_uri:
            results.append({"filename": file_name, "transcription": "[UPLOAD FAILED]"})
            continue

        # Start job
        if not start_transcription_job(job_name, s3_uri, 'mdt-output'):
            results.append({"filename": file_name, "transcription": "[JOB START FAILED]"})
            continue

        # Wait for job completion
        job_info = wait_for_completion(job_name)
        if not job_info:
            results.append({"filename": file_name, "transcription": "[JOB TIMEOUT OR ERROR]"})
            continue

        if job_info["TranscriptionJobStatus"] == "COMPLETED":
            transcript_uri = job_info["Transcript"]["TranscriptFileUri"]
            #text = get_transcript_from_uri(transcript_uri)
            text = get_transcript_from_s3_bucket(transcript_uri, S3_OUTPUT)

        else:
            failure_reason = job_info.get("FailureReason", "Unknown error")
            text = f"[FAILED: {failure_reason}]"

        results.append({"filename": file_name, "transcription": text})

        '''if len(results)%10== 0:

            # --- Save results ---
            try:
                df = pd.DataFrame(results)
                df.to_csv(OUTPUT_CSV, index=False, mode="a", header=not os.path.exists("checkpoint.csv"))
                results = []  # clear saved entries
            except Exception as e:
                print(f"❌ Error saving CSV: {e}")'''

    print("✅ Saving final results...")
    df = pd.DataFrame(results)
    df =df[df['filename']!='filename']
    print (df.shape)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"All transcriptions saved to {OUTPUT_CSV}")

100%|████████████████████████████████████████████████████████████████████████████████| 127/127 [29:44<00:00, 14.05s/it]


✅ Saving final results...
(127, 2)
All transcriptions saved to outputs/AWS_Transcript_2/MDT_meeting_full_case1.csv


100%|████████████████████████████████████████████████████████████████████████████████| 127/127 [30:45<00:00, 14.53s/it]


✅ Saving final results...
(127, 2)
All transcriptions saved to outputs/AWS_Transcript_2/MDT_meeting_full_case2.csv


100%|████████████████████████████████████████████████████████████████████████████████| 127/127 [41:29<00:00, 19.60s/it]


...

100%|████████████████████████████████████████████████████████████████████████████████| 127/127 [27:22<00:00, 12.93s/it]


✅ Saving final results...
(127, 2)
All transcriptions saved to outputs/AWS_Transcript_2/MDT_meeting_full_case8.csv


100%|████████████████████████████████████████████████████████████████████████████████| 128/128 [29:20<00:00, 13.76s/it]


✅ Saving final results...
(127, 2)
All transcriptions saved to outputs/AWS_Transcript_2/MDT_meeting_full_case9.csv


 59%|███████████████████████████████████████████████▊                                 | 75/127 [22:41<11:49, 13.64s/it]

In [6]:
 df.to_csv('outputs/AWS_Transcript_2/MDT_meeting_full_case1.csv', index=False, mode="a", header=not os.path.exists("checkpoint.csv"))